In [2]:
import sys
from pathlib import Path

# To import the variables in config.py
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f 
from config import OFF_CSV_PATH, OFF_CSV_URL
import urllib.request

In [4]:
spark = SparkSession.builder.appName(
        "lakehouse-off").master("local[*]").getOrCreate() 

In [5]:
if not OFF_CSV_PATH.exists():
    # Create the data/raw dir
    OFF_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    # Download the csv file only when it doesn't exist 
    urllib.request.urlretrieve(OFF_CSV_URL, OFF_CSV_PATH)

In [6]:
# Verify the file path and its size. 
print(OFF_CSV_PATH.exists())
print(OFF_CSV_PATH.stat().st_size)

True
1261161707


In [26]:
df = spark.read.csv(
    str(OFF_CSV_PATH),
    sep = "\t",
    header=True,
    multiLine=True,
    quote='"',
    escape='"',
    encoding="utf-8"
    )

In [27]:
df.show(10)

+------------+--------------------+--------------------+----------+--------------------+---------------+----------------------+----------------+--------------+---------------------+--------------------+------------------------+------------+--------+---------+--------------+------------+--------------+--------------+-----------------+--------------+--------------------+--------------------+--------------------+-------+------------+----------+--------------------+-------------------------+--------------------+--------------------+--------------------+---------+--------------+------------------------+------+-----------+---------------+------+----------------+-------------------+-------------+--------------------+--------------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+---------------+--------------------+----------------+----------------+----------+-------------+----------------

In [28]:
df.printSchema()

root
 |-- code: string (nullable = true)
 |-- url: string (nullable = true)
 |-- creator: string (nullable = true)
 |-- created_t: string (nullable = true)
 |-- created_datetime: string (nullable = true)
 |-- last_modified_t: string (nullable = true)
 |-- last_modified_datetime: string (nullable = true)
 |-- last_modified_by: string (nullable = true)
 |-- last_updated_t: string (nullable = true)
 |-- last_updated_datetime: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- abbreviated_product_name: string (nullable = true)
 |-- generic_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- packaging: string (nullable = true)
 |-- packaging_tags: string (nullable = true)
 |-- packaging_en: string (nullable = true)
 |-- packaging_text: string (nullable = true)
 |-- brands: string (nullable = true)
 |-- brands_tags: string (nullable = true)
 |-- brands_en: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- categories_tags: s

Most of the columns present are in "String", 

In [21]:
print(f"There are {len(df.columns)} columns in the whole data set.")
print(f"There are {df.count()} rows in the whold data set.")

There are 210 columns in the whole data set.
There are 4475958 rows in the whold data set.


In [17]:
numeric_cols = [col for col in df.columns if "100g" in col]
len(numeric_cols)
print(numeric_cols)

['energy-kj_100g', 'energy-kcal_100g', 'energy_100g', 'energy-from-fat_100g', 'fat_100g', 'saturated-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g', 'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g', 'dihomo-gamma-linolenic-acid_100g', 'oleic-acid_100g', 'elaidic-acid_100g', 'gondoic-acid_100g', 'mead-acid_100g', 'erucic-acid_100g', 'nervonic-acid_100g', 'trans-fat_100g', 'cholesterol_100g', 'carbohydrates_100g', 'sugars_100g', 'added-sugars_100g', 'sucrose_100g', 'glucose_100g

#### Check the Null values in column level 

In [29]:
# create a sample DataFrame
df_sample = df.sample(fraction=0.01, seed=42)
total_sample = df_sample.count()

In [ ]:
# check the number of Null values presenting in each columns 
df_sample.select([
f.count(f.when(f.col(c).isNull(), c)).alias(c)
    for c in df_sample.columns
]).show()

+----+---+-------+---------+----------------+---------------+----------------------+----------------+--------------+---------------------+------------+------------------------+------------+--------+---------+--------------+------------+--------------+------+-----------+---------+----------+---------------+-------------+-------+------------+----------+--------------------+-------------------------+------+-----------+---------+---------+--------------+------------------------+------+-----------+---------------+------+---------+--------------+------------+----------------+----------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+--------------+------------+----------------+----------------+----------+-------------+-------------+-----------+----------------+--------------+------+-----------+---------+-----------+-------------------------+-------------------------+--------------------+

In [31]:
# check the percentage of Null values in each columns 
df_sample.select([
f.round(
    (f.count(f.when(f.col(c).isNull(), 1)) / total_sample), 2
).alias(c)
    for c in df_sample.columns
]).show()

+----+---+-------+---------+----------------+---------------+----------------------+----------------+--------------+---------------------+------------+------------------------+------------+--------+---------+--------------+------------+--------------+------+-----------+---------+----------+---------------+-------------+-------+------------+----------+--------------------+-------------------------+------+-----------+---------+---------+--------------+------------------------+------+-----------+---------------+------+---------+--------------+------------+----------------+----------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+--------------+------------+----------------+----------------+----------+-------------+-------------+-----------+----------------+--------------+------+-----------+---------+-----------+-------------------------+-------------------------+--------------------+

In [ ]:
# Check the Null Value in Row level

